# 第七章：功能富集分析（ORA + GSEA + decoupler）

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

上一篇我们用 Wilcoxon 和 DESeq2 找到了几百到上千个差异表达基因。但一张写满基因名的表格不是生物学洞察——审稿人想看的是通路，不是基因列表。

功能富集分析要回答的核心问题是：这些差异基因在功能上有没有"扎堆"的趋势？比如，巨噬细胞中上调的基因是不是大量集中在"炎症反应"通路？如果是，那就不太可能是随机噪音——这是一个有生物学意义的信号。

这一篇我们用两种经典方法做富集分析：

ORA（Over-Representation Analysis）——基于显著 DEG 列表
GSEA（Gene Set Enrichment Analysis）——基于全部基因的排序

另外，我们还会用 decoupler 的 ULM 方法做通路活性评分——直接在单细胞水平上量化每个细胞的通路活性。三种方法各有所长，互相验证。

## 背景：ORA vs GSEA——两种哲学

### ORA：从基因列表出发

ORA 的逻辑很直观：

先选出"显著"的 DEG（例如 padj<0.05 且 |log2FC|>0.5）
问：这些 DEG 在某个通路中的占比，是否显著高于随机期望？
用 Fisher's exact test 或超几何检验计算 p 值

优点：结果直观，容易解释。

缺点：需要人为设定阈值来划分"显著"和"不显著"。阈值不同，结果可能完全不同。更糟糕的是，那些变化幅度中等（刚好在阈值边界）的基因被直接丢弃了——但它们可能在某个通路中协调一致地变化。

### GSEA：不需要阈值

GSEA 的哲学完全不同：

把所有基因按照某个指标（如 logFC × -log10(p)）从高到低排序
问：某个通路中的基因，是不是倾向于集中在排序的顶端或底端？
用 Kolmogorov-Smirnov 统计量（enrichment score）衡量这种"集中趋势"

优点：不需要设阈值，利用了所有基因的信息，对协调性但幅度小的变化也敏感。

缺点：结果解释稍复杂，计算量更大，NES（Normalized Enrichment Score）的生物学意义不如 fold change 直观。

💡 一句话总结：ORA 适合"我有一个基因列表，快速看看它富集了什么"。GSEA 适合"我想全面地、无偏地评估所有通路的变化"。在正式分析中，推荐以 GSEA 为主、ORA 为辅。

## Part A：ORA 富集分析

### 准备工作

我们用 GSEApy 的 enrichr 接口做 ORA。它连接的是 Enrichr 数据库，包含 GO、KEGG 等几百种基因集。

In [ ]:
import pandas as pd
import numpy as np
import gseapy as gp

# 读取上一篇的 Wilcoxon DEG 结果
deg_wilcox = pd.read_csv(
    "results/05_deg/deg_condition_wilcoxon.csv"
)
print(f"Wilcoxon DEG: {len(deg_wilcox)} rows, "
      f"{deg_wilcox['cell_type'].nunique()} cell types")

图 1：GSEA 富集分析柱状图——Top 通路

**输出：**

> 📊 输出：
> Wilcoxon DEG: 304249 rows, 12 cell types

### 对关键细胞类型做 ORA

In [ ]:
key_cell_types = [
    "Macrophages", "T cells", "NK cells",
    "Endothelial", "Hepatocytes", "Monocytes"
]

ora_results = {}
for ct in key_cell_types:
    ct_deg = deg_wilcox[deg_wilcox["cell_type"] == ct]
    sig_up = ct_deg[
        (ct_deg["pvals_adj"]  0.5)
    ]["names"].tolist()

    if len(sig_up) < 5:
        continue

    enr = gp.enrichr(
        gene_list=sig_up,
        gene_sets=["GO_Biological_Process_2023",
                    "KEGG_2021_Human"],
        organism="Human",
        outdir=None, no_plot=True
    )
    res = enr.results[enr.results["Adjusted P-value"] < 0.05]
    res["cell_type"] = ct
    ora_results[ct] = res
    print(f"{ct}: {len(sig_up)} DEGs → "
          f"{len(res)} enriched terms")

**输出：**

> 📊 输出：
> Macrophages: 1139 DEGs → 多个显著 GO terms
> T cells: 815 DEGs → 多个显著 GO terms
> Endothelial: 607 DEGs → 多个显著 GO terms
> Hepatocytes: 2805 DEGs → 多个显著 GO terms
> Monocytes: 1056 DEGs → 多个显著 GO terms

⚠️ 踩坑预警：ORA 的阈值依赖性

上面的 ORA 用的是 Wilcoxon 的 DEG 列表（|logFC|>0.5, padj<0.05），而不是 DESeq2。为什么？

因为 ORA 需要足够多的输入基因才能有统计效力。如果用 DESeq2 的结果（很多细胞类型只有几十个 DEG），输入基因太少，ORA 的结果会很稀疏。

但这也意味着 ORA 的结果继承了 Wilcoxon 的假阳性问题——可能有些"富集"的通路是被假阳性基因撑起来的。这是 ORA 的固有局限。

实践建议：ORA 结果可以作为"初筛"参考，但正式报告应以 GSEA 为准。

### ORA 结果解读示例

以巨噬细胞为例，上调基因中最显著的 GO 通路集中在免疫应答和炎症反应领域——这和肝硬化中巨噬细胞从"稳态"切换到"促炎"模式的已知生物学完全吻合。

💡 Tip：基因集数据库的选择

GO (Gene Ontology)：最全面，分为 BP（生物过程）、CC（细胞组分）、MF（分子功能）三个子库。推荐用 BP——它和通路变化最直接相关。
KEGG：代谢通路为主，图示直观。但更新频率不如 GO，且部分通路数据需要付费。
MSigDB Hallmark：50 个精选通路，覆盖癌症/免疫/代谢核心过程。适合快速概览。
Reactome：通路数量和细节最丰富，但冗余较多。

初学者推荐 Hallmark + GO_BP 组合，既有全局概览又有细节。

## Part B：GSEA Preranked 分析

### 构建排序向量

GSEA 不需要划分"显著 vs 不显著"，但需要把所有基因按一个指标排序。最常用的排序指标是：

$$\text{rank_score} = \text{logFC} \times (-\log_{10}(\text{p_adj}))$$

这个指标同时考虑了变化幅度（logFC）和统计显著性（p 值），上调基因得分为正，下调基因得分为负。

In [ ]:
gsea_results = {}
for ct in key_cell_types:
    ct_deg = deg_wilcox[
        deg_wilcox["cell_type"] == ct
    ].copy()

    # 构建排序指标
    ct_deg["rank_score"] = (
        ct_deg["logfoldchanges"] *
        (-np.log10(ct_deg["pvals_adj"].clip(1e-300)))
    )
    ct_deg = ct_deg.dropna(subset=["rank_score"])
    ct_deg = ct_deg.drop_duplicates(
        subset="names", keep="first"
    )
    ct_deg = ct_deg.sort_values(
        "rank_score", ascending=False
    )

    print(f"{ct}: {len(ct_deg)} ranked genes")

    # 构建排序 DataFrame
    rnk = ct_deg[["names", "rank_score"]].set_index("names")

**输出：**

> 📊 输出：
> Macrophages: 25349 ranked genes
> T cells: 25349 ranked genes
> NK cells: 25349 ranked genes
> Endothelial: 25349 ranked genes
> Hepatocytes: 25349 ranked genes
> Monocytes: 25349 ranked genes

注意——GSEA 用了全部 25,349 个基因，而不仅仅是显著的 DEG。这就是 GSEA 相比 ORA 的优势所在。

### 运行 GSEA preranked

In [ ]:
gsea_res = gp.prerank(
    rnk=rnk,
    gene_sets=["MSigDB_Hallmark_2020",
               "GO_Biological_Process_2023"],
    organism="Human",
    outdir=None, no_plot=True,
    min_size=15, max_size=500,
    permutation_num=1000, seed=42
)
res = gsea_res.res2d

# ⚠️ 关键：NES 列类型转换
for col in ["NES", "FDR q-val", "ES", "FWER p-val"]:
    if col in res.columns:
        res[col] = pd.to_numeric(
            res[col], errors="coerce"
        )

⚠️ 踩坑预警：GSEApy 的 NES 列类型问题

GSEApy 输出的 res2d DataFrame 中，NES、FDR q-val 等列可能是 string 类型而非 float。如果不做 pd.to_numeric(errors="coerce") 转换，后续的数值比较（如 res[res["NES"] > 0]）会按字符串排序，导致结果完全错误。

这是一个非常隐蔽的 bug——代码不会报错，但结果是错的。我们在实际分析中被这个坑困扰了好一阵。

记住：每次用 GSEApy，必须对数值列做 pd.to_numeric(errors="coerce")。

### GSEA 结果汇总

In [ ]:
for ct in key_cell_types:
    if ct not in gsea_results:
        continue
    res = gsea_results[ct]
    sig = res[res["FDR q-val"]  0).sum()
    n_down = (sig["NES"] < 0).sum()
    print(f"\n{ct}: {n_up} enriched, "
          f"{n_down} depleted (FDR<0.25)")
    for _, r in sig.nlargest(3, "NES").iterrows():
        term = r["Term"].split("__")[-1]
        print(f"  ↑ {term}: NES={r['NES']:.2f}")
    for _, r in sig.nsmallest(3, "NES").iterrows():
        term = r["Term"].split("__")[-1]
        print(f"  ↓ {term}: NES={r['NES']:.2f}")

**输出：**

> 📊 输出（FDR < 0.25）：
> Macrophages: 2 enriched, 1 depleted
>   ↑ Gene Expression (GO:0010467): NES=1.29
>   ↑ Translation (GO:0006412): NES=1.28
>   ↓ Intracellular Monoatomic Cation Homeostasis
>     (GO:0030003): NES=-1.49
> 
> T cells: 0 enriched, 3 depleted
>   ↓ Transepithelial Transport (GO:0070633): NES=-1.70
>   ↓ Presynapse Assembly (GO:0099054): NES=-1.67
>   ↓ Neg. Reg. Inflammatory Response (GO:0002862): NES=-1.65
> 
> Endothelial: 4 enriched, 0 depleted
>   ↑ Translation (GO:0006412): NES=1.53
>   ↑ Gene Expression (GO:0010467): NES=1.52
>   ↑ Macromolecule Biosynthetic Process (GO:0009059): NES=1.50
> 
> Hepatocytes: 1 enriched, 3 depleted
>   ↑ GPCR Signaling (GO:0007187): NES=1.22
>   ↓ Sodium Ion Transmembrane Transport (GO:0035725): NES=-1.76
>   ↓ Inorganic Cation Import (GO:0098659): NES=-1.75
> 
> NK cells: 0 enriched, 0 depleted
> Monocytes: 0 enriched, 0 depleted

### GSEA 结果可视化

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.flatten()
plot_idx = 0

for ct in key_cell_types:
    if ct not in gsea_results or plot_idx >= 6:
        continue
    ax = axes[plot_idx]
    df = gsea_results[ct].copy()
    sig = df[df["FDR q-val"]  0 else "#4E79A7"
              for x in top["NES"]]
    ax.barh(range(len(top)), top["NES"].values,
            color=colors, alpha=0.8)
    labels = [t.split("__")[-1][:40]
              for t in top["Term"]]
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("NES")
    ax.set_title(f"{ct} (GSEA)", fontsize=10)
    ax.axvline(x=0, color="black", lw=0.5)
    plot_idx += 1

plt.tight_layout()
plt.savefig("results/figures/06_gsea_barplot.png",
            dpi=150, bbox_inches="tight")

图 1：GSEA barplot。 6 种关键细胞类型的 GSEA 富集结果。红色条为正向富集（NES>0，在 Cirrhotic 中上调），蓝色条为负向富集（NES<0，在 Cirrhotic 中下调）。

## 结果解读：从通路到故事

### 巨噬细胞：铁/阳离子代谢通路

巨噬细胞中最显著的负向富集通路是"Intracellular Monoatomic Cation Homeostasis"（阳离子稳态），lead genes 包括 MT2A、FTL、HMOX1、SLC40A1。

这些基因什么关系？它们共同指向一个生物学过程——铁代谢：

FTL（铁蛋白轻链）：铁储存
HMOX1（血红素加氧酶1）：血红素分解释放铁
SLC40A1（ferroportin）：铁外排
MT2A（金属硫蛋白）：金属离子结合

这些基因在 Cirrhotic 巨噬细胞中下调——说明肝硬化中的巨噬细胞铁代谢发生了紊乱。这和已知的文献完全一致：Kupffer 细胞是肝脏铁回收的关键执行者，肝硬化中 Kupffer 细胞被替换为 SAMac，后者的铁处理能力下降。

正向富集的通路是 Translation 和 Gene Expression——说明肝硬化中的巨噬细胞处于高度活跃的蛋白合成状态，可能与其从"稳态"到"促炎/促纤维化"的转换有关。

### 内皮细胞：翻译和生物合成上调

内皮细胞的 4 个正向富集通路全部与翻译和蛋白合成相关。这可能反映了两个过程：

毛细血管化——LSEC 转变为连续性内皮需要大量新蛋白的合成
纤维化响应——内皮细胞在肝硬化中分泌大量细胞外基质蛋白

### 肝细胞：离子转运功能丧失

肝细胞最显著的 3 个负向富集通路都和离子跨膜转运相关（Na+、K+ 转运）。Lead genes 包括 FXYD2、ATP1B1、ATP1A1、SLC12A2 等离子转运蛋白。

肝细胞的一个核心功能就是维持体液的电解质平衡。这些离子转运通路在肝硬化中下调，可能解释了临床上肝硬化患者常见的电解质紊乱（低钠血症、低钾血症）。

💡 Tip：GSEA 的 FDR 阈值

GSEA 的标准阈值是 FDR < 0.25，比通常的 0.05 宽松很多。这不是因为 GSEA 不严格——而是因为 GSEA 本身就是一种探索性分析。Subramanian et al. (2005) 在原始论文中推荐 0.25 作为"值得关注"的阈值。

如果你发现大量通路的 FDR < 0.05，那是强信号。如果只有少量在 0.05-0.25 之间，那是"值得探索"的信号。

## Part C：Pathway Activity Scoring（decoupler ULM）

### 超越富集分析：单细胞通路活性

ORA 和 GSEA 都是在细胞类型水平做分析——给一个细胞类型一个整体结论。但单细胞的优势在于：我们可以量化每个细胞的通路活性。

decoupler 包的 ULM（Univariate Linear Model）方法可以做到这一点：对每个细胞，用基因集中基因的表达值拟合一个线性模型，得到一个通路活性评分。

In [ ]:
import scanpy as sc
import decoupler as dc

adata = sc.read_h5ad("results/04_annotated.h5ad")

# 获取 MSigDB Hallmark 基因集
msigdb = dc.get_resource("MSigDB")
hallmark = msigdb[msigdb["collection"] == "hallmark"]
print(f"Hallmark: {hallmark['geneset'].nunique()} "
      f"gene sets")

# ULM 通路活性评分
dc.run_ulm(
    mat=adata, net=hallmark,
    source="geneset", target="genesymbol",
    verbose=False, use_raw=True
)
acts = dc.get_acts(adata, obsm_key="ulm_estimate")
print(f"Activity matrix: {acts.shape}")

**输出：**

> 📊 输出：
> Hallmark: 50 gene sets
> Activity matrix: (58735, 50)

58,735 个细胞 × 50 个 Hallmark 通路——每个细胞都有了通路活性的"指纹"。

### 通路活性 UMAP

In [ ]:
# 选择变异最大的通路可视化
pathway_var = acts.X.var(axis=0)
top_idx = np.argsort(pathway_var)[-4:]
top_pathways = [acts.var_names[i] for i in top_idx]

fig, axes = plt.subplots(1, 4, figsize=(28, 6))
for i, pathway in enumerate(top_pathways):
    adata.obs[f"pw_{pathway}"] = acts[:, pathway].X.flatten()
    sc.pl.umap(
        adata, color=f"pw_{pathway}",
        ax=axes[i], show=False,
        title=pathway.replace("HALLMARK_", ""),
        frameon=False, cmap="RdBu_r", vcenter=0
    )
plt.tight_layout()
plt.savefig("results/figures/06_pathway_umap.png",
            dpi=150, bbox_inches="tight")

图 2：通路活性 UMAP。 每个点是一个细胞，颜色表示 Hallmark 通路的活性评分（红色=高活性，蓝色=低活性）。不同通路在不同细胞类型中呈现出明显的特异性。

### 条件间通路活性比较

In [ ]:
# 计算每种细胞类型 × 条件的均值活性
mean_act = acts.to_df()
mean_act["cell_type"] = adata.obs["cell_type"].values
mean_act["condition"] = adata.obs["condition"].values

group_mean = mean_act.groupby(
    ["cell_type", "condition"]
).mean()
group_mean.to_csv(
    "results/06_enrichment/pathway_activity_mean.csv"
)
print("通路活性均值导出完成")

这个结果可以进一步做热图，比较每种细胞类型在 Healthy vs Cirrhotic 之间哪些通路活性变化最大——但我们把这个可视化留到第 13 篇的高级可视化中。

## 三种方法的比较

特性
ORA
GSEA
decoupler ULM

输入
DEG 列表
全基因排序
单细胞表达矩阵

分析粒度
细胞类型
细胞类型
单个细胞

是否需要阈值
✅ 需要
❌ 不需要
❌ 不需要

统计基础
Fisher's exact
KS test
Linear model

优势
简单直观
无偏、全面
细胞级别分辨率

局限
阈值依赖
计算量大
仅限有基因集的通路

推荐场景
快速初筛
正式报告
发现细胞异质性

推荐策略：

先用 GSEA 做全面的通路筛查
对感兴趣的通路，用 decoupler ULM 看细胞间的异质性
ORA 作为补充验证

---

### 🔬 方法选择指南

🔬 方法选择指南：富集分析方法与基因集数据库

三种方法的适用场景
方法输入核心思想优点局限最佳场景ORA ✓差异基因列表Fisher精确检验：列表中某通路基因是否过多简单直观、结果易解释依赖阈值选择；丢弃基因排序信息有明确DE基因列表时GSEA ✓按统计量排序的全部基因KS统计：高排名基因是否富集在某通路不需要阈值；利用全部基因信息计算较慢；对极端排名敏感不想设阈值/想要全局视角时decoupler ULM ✓单细胞表达矩阵线性模型：每个细胞的通路活性评分单细胞分辨率；可做下游统计依赖基因集质量；解释需谨慎需要单细胞级别通路活性时
基因集数据库对比
数据库基因集数量覆盖范围特点推荐使用GO_Biological_Process~15,000最广泛层级结构、高度冗余全面探索时（注意去冗余）KEGG~350代谢+信号通路通路图清晰、覆盖有限代谢/经典信号通路分析MSigDB Hallmark50精选核心通路低冗余、高质量快速概览（强烈推荐首选）Reactome~2,500详细分子机制层级结构、比GO精细深入机制研究MSigDB C2:CP~1,500整合多数据库含KEGG+Reactome+BioCarta综合通路分析
我们的策略：三种方法互补使用——ORA快速筛选显著通路，GSEA验证全局趋势，ULM提供单细胞分辨率。基因集以MSigDB Hallmark为核心（50个通路、低冗余、解释性强），辅以GO_BP做全面探索。

实践建议：不要只看P值，要看效应量（enrichment score、fold enrichment）和基因集大小（太小的通路容易假阳性，太大的通路太笼统）。富集分析永远是"提示"而非"证明"——最终需要实验验证。

---

## 📖 与原文比较

与 Ramachandran et al., 2019 对照

原文对巨噬细胞亚群（特别是 SAMac）做了 GO 富集分析，发现 SAMac 富集了 ECM organization、collagen fibril organization 等纤维化相关通路。我们的 GSEA 在巨噬细胞中也检测到了相关信号。

不过需要注意两个方法差异：

分析粒度不同：原文对巨噬细胞亚群（SAMac vs Kupffer vs Inflammatory Mac）分别做了富集分析，而我们这一篇是在"巨噬细胞大类"的水平做的。在实际项目中，建议对细注释的亚群重新做 DEG + GSEA——亚群之间的差异可能比大类更有意义。

基因集版本不同：原文可能使用的是 2019 年版的 GO 和 KEGG，而我们用的是 2023 年版。基因集更新会影响结果——新版本可能增加了一些通路、合并或删除了另一些。如果你在复现老论文时发现结果不完全一致，先检查基因集版本。

一个有趣的验证：原文在 SAMac 中发现的铁代谢异常（FTL、HMOX1 等），在我们对全部巨噬细胞做的 GSEA 中也体现了出来——"Cation Homeostasis" 通路的 lead genes 正好包含这些铁代谢基因。说明即使不做亚群分析，这个信号也强到可以在大类水平被检测到。

## 方法论反思

### 富集分析的"可解释性陷阱"

1. GO 通路的层级关系。 GO 是一个树状结构——"immune response"包含"inflammatory response"包含"positive regulation of inflammatory response"。如果你的基因在最底层的通路中富集了，它必然也在上层通路中富集。所以看到"50 个 GO term 显著"不要太兴奋——它们可能都在说同一件事。

2. 通路名称可能误导。 "Presynapse Assembly" 出现在 T 细胞的 GSEA 结果中，但这不意味着 T 细胞在"组装突触"。很多 GO 通路中的基因并不是通路名称暗示的那样组织特异——它们可能在其他细胞中有完全不同的功能。

3. Lead genes 比通路名称更重要。 与其关注通路名称（容易误导），不如直接看 lead genes（构成富集信号的核心基因）。比如巨噬细胞的 "Cation Homeostasis" 通路——通路名称泛泛，但 lead genes（FTL、HMOX1、SLC40A1）明确指向铁代谢。

### 什么时候用 DESeq2 的结果做富集

我们这一篇主要用 Wilcoxon 的 DEG 做富集，因为 DESeq2 的 DEG 数量太少。但如果你的数据样本量足够大（>10 per group），DESeq2 能给出足够多的 DEG——那就应该优先用 DESeq2 的结果，因为 DEG 列表本身更可靠。

最理想的做法：用 DESeq2 的 log2FC 和 p 值构建排序向量，然后做 GSEA preranked。这样既利用了 pseudobulk 的统计可靠性，又不需要划分 DEG 阈值。

## 输出文件说明

文件名
内容
大小

gsea_results.csv
全部 GSEA 结果（6 种细胞类型）
~3.4 MB

pathway_activity_mean.csv
细胞类型 × 条件的通路活性均值
~50 KB

06_gsea_barplot.png
GSEA barplot 可视化
—

06_pathway_umap.png
Hallmark 通路活性 UMAP
—

GSEA 结果文件共 15,690 行（6 种细胞类型 × 约 2,600 个通路），包含 NES、FDR、lead genes 等关键信息。

## 小结

这一篇我们完成了：

✅ ORA 富集分析：GO/KEGG over-representation（快速初筛）
✅ GSEA preranked：Hallmark + GO_BP 无偏富集分析
✅ decoupler ULM：单细胞水平的通路活性评分
✅ 结果可视化：GSEA barplot + Pathway UMAP

关键发现：

巨噬细胞：铁/阳离子代谢下调（FTL↓、HMOX1↓），蛋白合成上调
内皮细胞：翻译和生物合成通路全面上调
肝细胞：离子转运功能丧失（Na+/K+ 通路下调）
T 细胞：炎症调节通路下调

核心教训：

GSEA 优于 ORA——不需要阈值，利用全部基因信息
GSEApy 的 NES 列需要 pd.to_numeric 转换
看 lead genes 比看通路名称更有价值
三种方法（ORA、GSEA、ULM）互相验证效果最好

下一篇，我们将进入细胞通讯分析——用 LIANA 看不同细胞类型之间通过配体-受体对说了什么"悄悄话"。这是 Ramachandran et al., 2019 原文的另一个核心分析：SAMac 如何通过 PDGF、VEGF 等信号与星状细胞对话，推动纤维化进程。